# PyCAM-SIMA Python-owned FKESSLER Notebook

This is the single maintained PyCAM-SIMA Notebook. A normal one-process Jupyter kernel controls 24 Python MPI workers through the existing authenticated socket/PBS controller.

There is one model path: `start()` performs pure-Python initialization, every persistent state array is owned by NumPy, and the Fortran shared library contains only stateless numerical kernels. The old cam_init/cam_run wrapper backend has been removed.

## Runtime architecture

```text
Jupyter controller
        │ authenticated socket / PBS
        ▼
24 Python MPI workers (one process per rank)
        ├── read YAML and atm_in
        ├── build cubed-sphere grid and rank-local partition
        ├── allocate every persistent NumPy array
        ├── read vertical coordinates and generate DCMIP2016 state
        ├── initialize clock, constituents, and parameters
        ├── communicate through mpi4py
        └── Python StatePool: T/U/V/Q/PS, tendencies, grid, process state
                          │ explicit zero-copy array arguments
                          ▼
              stateless Fortran numerical kernel .so
```

Rank 0 reads fixed NetCDF inputs and broadcasts them with `mpi4py`; no CAM initialization or driver routine is called. The shared library is loaded before initialization, but its ABI and numerical functions are first touched only at an explicit computational phase.

## 1. Fixed v1 configuration

The model is intentionally fixed to CAM-SIMA `f8daa568eae2696b7c4ebff7768f02f5d097d9df`, FKESSLER, ne3np4.pg3, L30, 24 MPI ranks, one thread per rank, a 1800 s timestep, and the DCMIP2016 moist baroclinic wave.

In [1]:
from datetime import datetime
from pathlib import Path
import json
import os
import shutil

import pycam_sima
from pycam_sima import ModelConfig, NotebookSession
from pycam_sima.model.validation import (
    compare_history_directories,
    compare_history_files,
)

repo = Path('/glade/work/ruitong/pycam-sima')
scratch = Path(os.environ.get('SCRATCH', '/glade/derecho/scratch/ruitong'))
config_path = repo / 'configs/fkessler_model.yaml'
config = ModelConfig.from_yaml(config_path)
reference_atm_in = (
    repo / 'reference/cases/FKESSLER_ne3pg3_gnu_24x50/CaseDocs/atm_in'
)
oracle_path = os.environ.get('PYCAM_SIMA_ORACLE_DIR')
oracle_run = Path(oracle_path).expanduser().resolve() if oracle_path else None

stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
run_dir = scratch / 'pycam-sima/notebook_trials' / f'model-{stamp}' / 'run'
run_dir.mkdir(parents=True, exist_ok=False)
shutil.copy2(reference_atm_in, run_dir / 'atm_in')
history_dir = run_dir / 'history'

print('pycam_sima', pycam_sima.__version__)
print('run directory:', run_dir)
config.as_dict()

pycam_sima 0.7.1
run directory: /glade/derecho/scratch/ruitong/pycam-sima/notebook_trials/model-20260721-154813/run


{'source_revision': 'f8daa568eae2696b7c4ebff7768f02f5d097d9df',
 'source_root': '/glade/work/ruitong/pycam-sima/external/CAM-SIMA',
 'physics_suite': 'kessler',
 'grid': 'ne3np4.pg3',
 'ne': 3,
 'np': 4,
 'fv_nphys': 3,
 'pver': 30,
 'mpi_size': 24,
 'threads_per_rank': 1,
 'dt_seconds': 1800,
 'stop_n': 50,
 'calendar': 'NO_LEAP',
 'run_type': 'startup',
 'analytic_ic_type': 'moist_baroclinic_wave_dcmip2016',
 'pertlim': 0.0,
 'case_name': 'pycam_sima_legacy_oracle_f8daa568',
 'atm_in': 'atm_in',
 'history_enabled': True}

## 2. Start 24 Python MPI workers

On a Derecho login-node kernel, `start()` submits a PBS worker; inside an allocation it launches locally. It returns in state `INITIALIZED` before any model/kernel function or ABI version function has been called.

In [2]:
if 'model' in globals() and model.running:
    model.close()

model = NotebookSession(
    config_path,
    run_dir=run_dir,
    history_dir=history_dir,
    python_executable=repo / '.venv/bin/python',
    log_path=run_dir / 'mpi-worker.log',
)
model.start()
assert model.initialized_native_calls == 0
assert model.initialized_abi_checked is False
print({
    'mode': model.launch_mode_used,
    'job': model.job_id,
    'ranks': model.ranks,
    'fields_and_aliases': len(model.field_names),
    'phase_status': model.phase_status,
    'scheme_status': model.scheme_status,
})

PyCAM-SIMA PBS worker submitted as 6835174.desched1; waiting for 24 MPI ranks ...
{'mode': 'pbs', 'job': '6835174.desched1', 'ranks': 24, 'fields_and_aliases': 316, 'phase_status': {'runtime': 'model', 'state': 'INITIALIZED', 'last_phase': None, 'last_scheme': None, 'last_scheme_group': None, 'next_phase': None, 'sequence_safe': True, 'step': 0, 'native_nstep': 0}, 'scheme_status': {'last_scheme': None, 'last_scheme_group': None, 'sequence_safe': True, 'groups': ('physics_before_coupler', 'physics_after_coupler'), 'plan': {'sequence_safe': True, 'schemes': [{'group': 'physics_before_coupler', 'source_group': 'physics_before_coupler', 'name': 'calc_exner', 'enabled': True}, {'group': 'physics_before_coupler', 'source_group': 'physics_before_coupler', 'name': 'temp_to_potential_temp', 'enabled': True}, {'group': 'physics_before_coupler', 'source_group': 'physics_before_coupler', 'name': 'calc_dry_air_ideal_gas_density', 'enabled': True}, {'group': 'physics_before_coupler', 'source_group'

## 3. Inspect Python-owned fields

`parameters.field(name)` returns a typed remote handle. `get()` transfers a rank-local copy to the Notebook; `stats()` computes a compact summary in the MPI worker. Canonical state metadata records owner, intent, dimensions, units, lifetime, category, and alias status.

In [3]:
temperature_field = model.parameters.air_temperature
print(temperature_field.info)
print(temperature_field.stats(rank=0))
temperature_rank0 = temperature_field.get(rank=0)
temperature_rank0

{'standard_name': 'air_temperature', 'shape': (4, 4, 30, 3, 3), 'dtype': '<f8', 'dimensions': ('np', 'np', 'pver', 'nelem_local', 'ntime'), 'intent': 'inout', 'owner': 'python', 'lifetime': 'persistent', 'category': 'se_state', 'units': 'K', 'writable': True, 'alias': False}
{'rank': 0, 'shape': (4, 4, 30, 3, 3), 'dtype': '<f8', 'min': 149.84337724047597, 'max': 306.20463514548004, 'mean': 237.25174410239765}


array([[[[[150.29306756, 150.29306756, 150.29306756],
          [149.8485939 , 149.8485939 , 149.8485939 ],
          [149.8485939 , 149.8485939 , 149.8485939 ]],

         [[159.12576481, 159.12576481, 159.12576481],
          [158.23216121, 158.23216121, 158.23216121],
          [158.23216121, 158.23216121, 158.23216121]],

         [[167.46729417, 167.46729417, 167.46729417],
          [165.97080678, 165.97080678, 165.97080678],
          [165.97080678, 165.97080678, 165.97080678]],

         ...,

         [[290.36856087, 290.36856087, 290.36856087],
          [303.96388565, 303.96388565, 303.96388565],
          [303.96388565, 303.96388565, 303.96388565]],

         [[291.32212781, 291.32212781, 291.32212781],
          [305.08283314, 305.08283314, 305.08283314],
          [305.08283314, 305.08283314, 305.08283314]],

         [[292.13216035, 292.13216035, 292.13216035],
          [306.03088151, 306.03088151, 306.03088151],
          [306.03088151, 306.03088151, 306.03088151]]],



In [4]:
parameter_summary = model.parameters.describe()
assert parameter_summary['runtime']['state_owner'] == 'python'
assert all(model.field_info(name)['owner'] == 'python' for name in model.field_names)
print('all fields and zero-copy aliases:', len(parameter_summary['all_fields']))
parameter_summary['key_fields']

all fields and zero-copy aliases: 316


{'air_temperature': {'standard_name': 'air_temperature',
  'shape': (4, 4, 30, 3, 3),
  'dtype': '<f8',
  'dimensions': ('np', 'np', 'pver', 'nelem_local', 'ntime'),
  'intent': 'inout',
  'owner': 'python',
  'lifetime': 'persistent',
  'category': 'se_state',
  'units': 'K',
  'writable': True,
  'alias': False},
 'zonal_wind': {'standard_name': 'zonal_wind',
  'shape': (4, 4, 30, 3, 3),
  'dtype': '<f8',
  'dimensions': ('np', 'np', 'pver', 'nelem_local', 'ntime'),
  'intent': 'inout',
  'owner': 'python',
  'lifetime': 'persistent',
  'category': 'se_state',
  'units': 'm s-1',
  'writable': True,
  'alias': False},
 'meridional_wind': {'standard_name': 'meridional_wind',
  'shape': (4, 4, 30, 3, 3),
  'dtype': '<f8',
  'dimensions': ('np', 'np', 'pver', 'nelem_local', 'ntime'),
  'intent': 'inout',
  'owner': 'python',
  'lifetime': 'persistent',
  'category': 'se_state',
  'units': 'm s-1',
  'writable': True,
  'alias': False},
 'surface_pressure': {'standard_name': 'surface_pre

## 4. Explicit nstep=0 preparation

`prepare_initial_step()` executes dynamics-to-physics mapping, physics timestep initialization, the FKESSLER before-coupler schemes, and writes nstep=0 history. Calling `model.step()` directly from `INITIALIZED` performs this preparation automatically.

In [5]:
status = model.prepare_initial_step()
print(status)
print('history files:', len(list(history_dir.glob('*.nc'))))

{'runtime': 'model', 'state': 'PRIMED', 'last_phase': 'physics_timestep_initial', 'last_scheme': 'physics_before_coupler.kessler_diagnostics', 'last_scheme_group': 'physics_before_coupler', 'next_phase': None, 'sequence_safe': True, 'step': 0, 'native_nstep': 0}
history files: 1


## 5. Inspect and run individual CCPP schemes

The Kessler CCPP suite is no longer a single opaque before/after block. All 19 `physics_before_coupler` schemes and all 5 `physics_after_coupler` schemes are separate Python interfaces. Every call is collective across all 24 workers, and every return checks that Python array addresses are unchanged.

In [6]:
{
    'model_phases': model.phase_names,
    'before_coupler': model.scheme_plan.describe('physics_before_coupler'),
    'after_coupler': model.scheme_plan.describe('physics_after_coupler'),
}

{'model_phases': ('dynamics_to_physics',
  'physics_timestep_initial',
  'physics_to_dynamics',
  'scale_physics_forcing',
  'apply_cam_forcing',
  'apply_cam_forcing_substep_2',
  'initialize_prim_step',
  'se_first_rhs',
  'se_type4_rk',
  'advance_hyperviscosity',
  'update_surface_dry_air_pressure',
  'advance_se_tracers',
  'advance_fvm_tracers',
  'vertical_remap_se',
  'vertical_remap_fvm',
  'compute_final_omega',
  'update_time_levels',
  'physics_timestep_final'),
 'before_coupler': [{'order': 1,
   'key': 'physics_before_coupler.calc_exner',
   'name': 'calc_exner',
   'group': 'physics_before_coupler',
   'execution_group': 'physics_before_coupler',
   'source_group': 'physics_before_coupler',
   'category': 'thermodynamics',
   'description': 'calculate the Exner function',
   'implementation': 'python',
   'required': True,
   'enabled': True},
  {'order': 2,
   'key': 'physics_before_coupler.temp_to_potential_temp',
   'name': 'temp_to_potential_temp',
   'group': 'physi

In [9]:
#Example for a fresh scheme-by-scheme session:
model.run_phase('dynamics_to_physics')
model.run_phase('physics_timestep_initial')
model.run_scheme('calc_exner', group='physics_before_coupler')
exner_after_scheme = model.parameters.field('exner_function').get(rank=0)
print('exner_function after calc_exner scheme:', exner_after_scheme)
model.run_scheme('kessler', group='physics_before_coupler')
state_after_kessler = model.parameters.physics_air_temperature.get(rank=0)

#Intentional non-validated control experiments require unsafe=True:
model.scheme_plan.disable('kessler_diagnostics', unsafe=True)
model.scheme_plan.move('kessler', after='kessler_update', unsafe=True)
model.scheme_plan.move('kessler', to_group='physics_after_coupler', unsafe=True)
model.scheme_plan.reset()  # restore the BFB-validated XML order

model.scheme_plan.sequence_safe

exner_function after calc_exner scheme: [[0.20104084 0.24798687 0.29746738 0.3469965  0.39363494 0.43569715
  0.4715659  0.4990785  0.52279661 0.5476421  0.57366838 0.60093174
  0.62942925 0.65923316 0.69046648 0.72319712 0.75749714 0.79344497
  0.8311291  0.86761511 0.89938183 0.92530266 0.94478617 0.95737294
  0.96608048 0.97402963 0.98119904 0.98756951 0.99312415 0.99785013]
 [0.20104084 0.24798687 0.29746738 0.3469965  0.39363494 0.43569715
  0.4715659  0.4990785  0.52279659 0.54764203 0.57366824 0.6009315
  0.62943968 0.65926318 0.69051474 0.72326222 0.75757746 0.79353812
  0.83123084 0.86771813 0.89947746 0.92538435 0.94485208 0.95742626
  0.96612393 0.9740635  0.98122391 0.9875862  0.99313364 0.99785319]
 [0.20104084 0.24798687 0.29746738 0.3469965  0.39363494 0.43569715
  0.4715659  0.4990785  0.52279658 0.54764197 0.57366814 0.60093132
  0.62944735 0.65928526 0.69055024 0.7233101  0.75763651 0.7936066
  0.83130561 0.86779383 0.89954768 0.92544432 0.94490044 0.95746539
  0.9661

True

## 6. Optional field modification

Prognostic, tendency, and process fields may be changed at a Python boundary without replacing their NumPy storage. Static grid/topology fields reject writes unless `unsafe=True` is explicit. Any numerical edit intentionally breaks BFB.

In [10]:
changed = temperature_field.get(rank=0)
changed[0, 0, 0, 0, 0] += 1.0e-6
temperature_field.set(changed, rank=0)
print(temperature_field.stats(rank=0))

#Static experiment, deliberately unsafe:
gll = model.parameters.field('gll_node').get(rank=0)
model.parameters.field('gll_node').set(gll, rank=0, unsafe=True)

{'rank': 0, 'shape': (4, 4, 30, 3, 3), 'dtype': '<f8', 'min': 149.84337724047597, 'max': 306.20463514548004, 'mean': 237.25174410262915}


## 7. Advance one complete 1800 s timestep

In [11]:
step = model.step()
print('completed step:', step)
print(temperature_field.stats(rank=0))
print('history files:', len(list(history_dir.glob('*.nc'))))

completed step: 1
{'rank': 0, 'shape': (4, 4, 30, 3, 3), 'dtype': '<f8', 'min': 149.84316729350525, 'max': 306.21195553507687, 'mean': 237.2515878168729}
history files: 2


## 8. Optional complete 50-step run

Run this only with untouched fields. `step(count)` remains one socket request while all 24 workers execute the same fixed plan.

In [12]:
# if model.current_step < config.stop_n:
#     model.step(config.stop_n - model.current_step)
# print(model.current_step, len(list(history_dir.glob('*.nc'))))

## 9. Finalize and optionally compare history

The model run is self-contained. To compare against output from the pinned external CAM-SIMA executable, set `PYCAM_SIMA_ORACLE_DIR` to its history directory before running the setup cell.

In [13]:
model.close()
print('closed:', run_dir)

closed: /glade/derecho/scratch/ruitong/pycam-sima/notebook_trials/model-20260721-154813/run


In [14]:
candidate_files = sorted(history_dir.glob('*.nc'))
if oracle_run is None:
    print('External comparison skipped; set PYCAM_SIMA_ORACLE_DIR to enable it')
else:
    for candidate in candidate_files:
        compare_history_files(oracle_run / candidate.name, candidate)
    print(f'BFB for all {len(candidate_files)} available model timestamps')

    if len(candidate_files) == 51:
        compare_history_directories(
            oracle_run, history_dir,
            expected_files=51, expected_numeric_variables=26,
        )
        print('FULL BFB: 50 steps, 51 timestamps, 26 numeric variables')

External comparison skipped; set PYCAM_SIMA_ORACLE_DIR to enable it


## 10. Optional Dask fan-out without a persistent socket

This optional experiment runs a common 24-rank base task, keeps its immutable checkpoint bundle behind a Dask Future, and starts independent control/no-Kessler/warm branches from that same state. Each branch is a new PBS/MPI task; no paused socket worker is reused. Set `run_dask_fanout = True` only when three additional develop-queue jobs are intended.

In [ ]:
from distributed import Client
from pycam_sima import BranchSpec, DaskExperimentClient, FieldEdit

run_dask_fanout = False
if run_dask_fanout:
    dask_run_root = (
        scratch / 'pycam-sima/dask_notebook_trials' / f'fanout-{stamp}'
    )
    with Client(
        processes=False, n_workers=3, threads_per_worker=1,
        dashboard_address=None,
    ) as dask_client:
        experiments = DaskExperimentClient(
            dask_client,
            config=config_path,
            initial_run_dir=run_dir,
            run_root=dask_run_root,
            python_executable=repo / '.venv/bin/python',
        )
        base_future = experiments.submit_base(
            BranchSpec('base', steps=1)
        )
        branch_futures = experiments.fork(
            base_future,
            (
                BranchSpec('control', steps=1),
                BranchSpec(
                    'no-kessler', steps=1,
                    disable_schemes=('kessler',),
                ),
                BranchSpec(
                    'warm', steps=1,
                    field_edits=(
                        FieldEdit('air_temperature', 'add', 1.0),
                    ),
                ),
            ),
        )
        dask_summaries = experiments.summaries(branch_futures)
    dask_summaries
else:
    print('Dask fan-out skipped; set run_dask_fanout = True to submit it')

## 11. Recorded full validation

The uninterrupted model gate is stored in `validation/fkessler_model_bfb.json`; Dask fan-out and the 25+25 checkpoint/restart BFB gate are stored in `validation/dask_checkpoint_fanout.json`.

In [15]:
evidence_paths = (
    repo / 'validation/fkessler_model_bfb.json',
    repo / 'validation/dask_checkpoint_fanout.json',
)
{
    path.name: json.loads(path.read_text())
    for path in evidence_paths if path.exists()
}

{'schema_version': 4,
 'validated_at': '2026-07-21T05:21:05-06:00',
 'repository': {'root': '/glade/work/ruitong/pycam-sima',
  'git_head_before_changes': '82b9eab5bf6bf14c5d78eb62ff4f55fc7b7d820c',
  'cross_group_parent_commit': 'db56591288d8635b61d745a9c9af81ad92a06cdf',
  'package_version': '0.7.1'},
 'architecture': {'source_packages': ['core', 'model', 'notebook'],
  'model_driver': 'pycam_sima.model.CAMDriver',
  'selectable_model_backends': 1,
  'runtime_selector_present': False,
  'cam_init_cam_run_wrapper_present': False,
  'legacy_package_present': False,
  'python_native_package_present': False,
  'maintained_notebooks': 1,
  'scheme_interfaces': 24,
  'before_coupler_schemes': 19,
  'after_coupler_schemes': 5},
 'runtime': 'model',
 'configuration': {'physics_suite': 'FKESSLER',
  'grid': 'ne3np4.pg3',
  'vertical_levels': 30,
  'mpi_ranks': 24,
  'threads_per_rank': 1,
  'timestep_seconds': 1800,
  'steps': 50,
  'history_samples': 51,
  'calendar': 'NO_LEAP',
  'run_type'